# hankel-hierarchy: Gates A–C on Kaggle

Settings (right panel): **Accelerator → GPU T4 x2 or P100**, **Internet → On** (needs a phone-verified account).

**First time:** run cells 1–4 interactively, then stop and check the smoke output.
**Real run:** use *Save Version → Save & Run All (Commit)*. It runs in the background for up to 12 h, and everything under `/kaggle/working` is kept as the version's output.

The smoke cell loads **no** checkpoint. Only cell 5 does.

In [ ]:
# 1. environment
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, sys; print(sys.version); print('torch', torch.__version__, 'cuda', torch.version.cuda)

In [ ]:
# 2. code at the latest pushed commit (the preregistration commit is its ancestor)
%cd /kaggle/working
!rm -rf hankel-hierarchy && git clone -q https://github.com/CC-Shivansh-Gupta/hankel-hierarchy
%cd hankel-hierarchy
!git log --oneline | cat
!bash reproduce.sh setup

In [ ]:
# 3. synthetic validation suites (§13), CPU only
!bash reproduce.sh test

In [ ]:
# 4. SMOKE: random-init network at torch seed 999, no checkpoint, no spectra
!bash reproduce.sh smoke

In [ ]:
# 5. THE DIAGNOSTIC RUN. Each task is resumable: finished seeds are skipped.
import subprocess
for task in ['walker-walk', 'cheetah-run', 'hopper-hop', 'quadruped-run', 'humanoid-walk', 'finger-spin']:
    subprocess.run(['bash', 'reproduce.sh', 'run', task], check=True)
    subprocess.run(['bash', '-c', 'cd /kaggle/working/hankel-hierarchy && tar czf /kaggle/working/results_partial.tgz results'], check=True)

In [ ]:
# 6. verdict + figures, then bundle everything for download
!bash reproduce.sh verdict
!cat results/verdict.json | head -40
!tar czf /kaggle/working/results_final.tgz results figures